In [1]:
import csv
import random
import re
import numpy as np

STOPWORDS = set(["0o", "0s", "3a", "3b", "3d", "6b", "6o", "a", "a1", "a2", "a3", "a4", "ab", "able", "about", "above", "abst", "ac", "accordance", "according", "accordingly", "across", "act", "actually", "ad", "added", "adj", "ae", "af", "affected", "affecting", "affects", "after", "afterwards", "ag", "again", "against", "ah", "ain", "ain't", "aj", "al", "all", "allow", "allows", "almost", "alone", "along", "already", "also", "although", "always", "am", "among", "amongst", "amoungst", "amount", "an", "and", "announce", "another", "any", "anybody", "anyhow", "anymore", "anyone", "anything", "anyway", "anyways", "anywhere", "ao", "ap", "apart", "apparently", "appear", "appreciate", "appropriate", "approximately", "ar", "are", "aren", "arent", "aren't", "arise", "around", "as", "a's", "aside", "ask", "asking", "associated", "at", "au", "auth", "av", "available", "aw", "away", "awfully", "ax", "ay", "az", "b", "b1", "b2", "b3", "ba", "back", "bc", "bd", "be", "became", "because", "become", "becomes", "becoming", "been", "before", "beforehand", "begin", "beginning", "beginnings", "begins", "behind", "being", "believe", "below", "beside", "besides", "best", "better", "between", "beyond", "bi", "bill", "biol", "bj", "bk", "bl", "bn", "both", "bottom", "bp", "br", "brief", "briefly", "bs", "bt", "bu", "but", "bx", "by", "c", "c1", "c2", "c3", "ca", "call", "came", "can", "cannot", "cant", "can't", "cause", "causes", "cc", "cd", "ce", "certain", "certainly", "cf", "cg", "ch", "changes", "ci", "cit", "cj", "cl", "clearly", "cm", "c'mon", "cn", "co", "com", "come", "comes", "con", "concerning", "consequently", "consider", "considering", "contain", "containing", "contains", "corresponding", "could", "couldn", "couldnt", "couldn't", "course", "cp", "cq", "cr", "cry", "cs", "c's", "ct", "cu", "currently", "cv", "cx", "cy", "cz", "d", "d2", "da", "date", "dc", "dd", "de", "definitely", "describe", "described", "despite", "detail", "df", "di", "did", "didn", "didn't", "different", "dj", "dk", "dl", "do", "does", "doesn", "doesn't", "doing", "don", "done", "don't", "down", "downwards", "dp", "dr", "ds", "dt", "du", "due", "during", "dx", "dy", "e", "e2", "e3", "ea", "each", "ec", "ed", "edu", "ee", "ef", "effect", "eg", "ei", "eight", "eighty", "either", "ej", "el", "eleven", "else", "elsewhere", "em", "empty", "en", "end", "ending", "enough", "entirely", "eo", "ep", "eq", "er", "es", "especially", "est", "et", "et-al", "etc", "eu", "ev", "even", "ever", "every", "everybody", "everyone", "everything", "everywhere", "ex", "exactly", "example", "except", "ey", "f", "f2", "fa", "far", "fc", "few", "ff", "fi", "fifteen", "fifth", "fify", "fill", "find", "fire", "first", "five", "fix", "fj", "fl", "fn", "fo", "followed", "following", "follows", "for", "former", "formerly", "forth", "forty", "found", "four", "fr", "from", "front", "fs", "ft", "fu", "full", "further", "furthermore", "fy", "g", "ga", "gave", "ge", "get", "gets", "getting", "gi", "give", "given", "gives", "giving", "gj", "gl", "go", "goes", "going", "gone", "got", "gotten", "gr", "greetings", "gs", "gy", "h", "h2", "h3", "had", "hadn", "hadn't", "happens", "hardly", "has", "hasn", "hasnt", "hasn't", "have", "haven", "haven't", "having", "he", "hed", "he'd", "he'll", "hello", "help", "hence", "her", "here", "hereafter", "hereby", "herein", "heres", "here's", "hereupon", "hers", "herself", "hes", "he's", "hh", "hi", "hid", "him", "himself", "his", "hither", "hj", "ho", "home", "hopefully", "how", "howbeit", "however", "how's", "hr", "hs", "http", "hu", "hundred", "hy", "i", "i2", "i3", "i4", "i6", "i7", "i8", "ia", "ib", "ibid", "ic", "id", "i'd", "ie", "if", "ig", "ignored", "ih", "ii", "ij", "il", "i'll", "im", "i'm", "immediate", "immediately", "importance", "important", "in", "inasmuch", "inc", "indeed", "index", "indicate", "indicated", "indicates", "information", "inner", "insofar", "instead", "interest", "into", "invention", "inward", "io", "ip", "iq", "ir", "is", "isn", "isn't", "it", "itd", "it'd", "it'll", "its", "it's", "itself", "iv", "i've", "ix", "iy", "iz", "j", "jj", "jr", "js", "jt", "ju", "just", "k", "ke", "keep", "keeps", "kept", "kg", "kj", "km", "know", "known", "knows", "ko", "l", "l2", "la", "largely", "last", "lately", "later", "latter", "latterly", "lb", "lc", "le", "least", "les", "less", "lest", "let", "lets", "let's", "lf", "like", "liked", "likely", "line", "little", "lj", "ll", "ll", "ln", "lo", "look", "looking", "looks", "los", "lr", "ls", "lt", "ltd", "m", "m2", "ma", "made", "mainly", "make", "makes", "many", "may", "maybe", "me", "mean", "means", "meantime", "meanwhile", "merely", "mg", "might", "mightn", "mightn't", "mill", "million", "mine", "miss", "ml", "mn", "mo", "more", "moreover", "most", "mostly", "move", "mr", "mrs", "ms", "mt", "mu", "much", "mug", "must", "mustn", "mustn't", "my", "myself", "n", "n2", "na", "name", "namely", "nay", "nc", "nd", "ne", "near", "nearly", "necessarily", "necessary", "need", "needn", "needn't", "needs", "neither", "never", "nevertheless", "new", "next", "ng", "ni", "nine", "ninety", "nj", "nl", "nn", "no", "nobody", "non", "none", "nonetheless", "noone", "nor", "normally", "nos", "not", "noted", "nothing", "novel", "now", "nowhere", "nr", "ns", "nt", "ny", "o", "oa", "ob", "obtain", "obtained", "obviously", "oc", "od", "of", "off", "often", "og", "oh", "oi", "oj", "ok", "okay", "ol", "old", "om", "omitted", "on", "once", "one", "ones", "only", "onto", "oo", "op", "oq", "or", "ord", "os", "ot", "other", "others", "otherwise", "ou", "ought", "our", "ours", "ourselves", "out", "outside", "over", "overall", "ow", "owing", "own", "ox", "oz", "p", "p1", "p2", "p3", "page", "pagecount", "pages", "par", "part", "particular", "particularly", "pas", "past", "pc", "pd", "pe", "per", "perhaps", "pf", "ph", "pi", "pj", "pk", "pl", "placed", "please", "plus", "pm", "pn", "po", "poorly", "possible", "possibly", "potentially", "pp", "pq", "pr", "predominantly", "present", "presumably", "previously", "primarily", "probably", "promptly", "proud", "provides", "ps", "pt", "pu", "put", "py", "q", "qj", "qu", "que", "quickly", "quite", "qv", "r", "r2", "ra", "ran", "rather", "rc", "rd", "re", "readily", "really", "reasonably", "recent", "recently", "ref", "refs", "regarding", "regardless", "regards", "related", "relatively", "research", "research-articl", "respectively", "resulted", "resulting", "results", "rf", "rh", "ri", "right", "rj", "rl", "rm", "rn", "ro", "rq", "rr", "rs", "rt", "ru", "run", "rv", "ry", "s", "s2", "sa", "said", "same", "saw", "say", "saying", "says", "sc", "sd", "se", "sec", "second", "secondly", "section", "see", "seeing", "seem", "seemed", "seeming", "seems", "seen", "self", "selves", "sensible", "sent", "serious", "seriously", "seven", "several", "sf", "shall", "shan", "shan't", "she", "shed", "she'd", "she'll", "shes", "she's", "should", "shouldn", "shouldn't", "should've", "show", "showed", "shown", "showns", "shows", "si", "side", "significant", "significantly", "similar", "similarly", "since", "sincere", "six", "sixty", "sj", "sl", "slightly", "sm", "sn", "so", "some", "somebody", "somehow", "someone", "somethan", "something", "sometime", "sometimes", "somewhat", "somewhere", "soon", "sorry", "sp", "specifically", "specified", "specify", "specifying", "sq", "sr", "ss", "st", "still", "stop", "strongly", "sub", "substantially", "successfully", "such", "sufficiently", "suggest", "sup", "sure", "sy", "system", "sz", "t", "t1", "t2", "t3", "take", "taken", "taking", "tb", "tc", "td", "te", "tell", "ten", "tends", "tf", "th", "than", "thank", "thanks", "thanx", "that", "that'll", "thats", "that's", "that've", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "thered", "therefore", "therein", "there'll", "thereof", "therere", "theres", "there's", "thereto", "thereupon", "there've", "these", "they", "theyd", "they'd", "they'll", "theyre", "they're", "they've", "thickv", "thin", "think", "third", "this", "thorough", "thoroughly", "those", "thou", "though", "thoughh", "thousand", "three", "throug", "through", "throughout", "thru", "thus", "ti", "til", "tip", "tj", "tl", "tm", "tn", "to", "together", "too", "took", "top", "toward", "towards", "tp", "tq", "tr", "tried", "tries", "truly", "try", "trying", "ts", "t's", "tt", "tv", "twelve", "twenty", "twice", "two", "tx", "u", "u201d", "ue", "ui", "uj", "uk", "um", "un", "under", "unfortunately", "unless", "unlike", "unlikely", "until", "unto", "uo", "up", "upon", "ups", "ur", "us", "use", "used", "useful", "usefully", "usefulness", "uses", "using", "usually", "ut", "v", "va", "value", "various", "vd", "ve", "ve", "very", "via", "viz", "vj", "vo", "vol", "vols", "volumtype", "vq", "vs", "vt", "vu", "w", "wa", "want", "wants", "was", "wasn", "wasnt", "wasn't", "way", "we", "wed", "we'd", "welcome", "well", "we'll", "well-b", "went", "were", "we're", "weren", "werent", "weren't", "we've", "what", "whatever", "what'll", "whats", "what's", "when", "whence", "whenever", "when's", "where", "whereafter", "whereas", "whereby", "wherein", "wheres", "where's", "whereupon", "wherever", "whether", "which", "while", "whim", "whither", "who", "whod", "whoever", "whole", "who'll", "whom", "whomever", "whos", "who's", "whose", "why", "why's", "wi", "widely", "will", "willing", "wish", "with", "within", "without", "wo", "won", "wonder", "wont", "won't", "words", "world", "would", "wouldn", "wouldnt", "wouldn't", "www", "x", "x1", "x2", "x3", "xf", "xi", "xj", "xk", "xl", "xn", "xo", "xs", "xt", "xv", "xx", "y", "y2", "yes", "yet", "yj", "yl", "you", "youd", "you'd", "you'll", "your", "youre", "you're", "yours", "yourself", "yourselves", "you've", "yr", "ys", "yt", "z", "zero", "zi", "zz", "amp", "dont"])

class MultinomialNaiveBayes:
    def __init__(self, nb_classes, nb_words, pseudocount):
        self.nb_classes = nb_classes
        self.nb_words = nb_words
        self.pseudocount = pseudocount

    def fit(self, X, Y):
        nb_examples = X.shape[0]
        self.priors = np.bincount(Y) / nb_examples
        occs = np.zeros((self.nb_classes, self.nb_words))
        for i in range(nb_examples):
            c = Y[i]
            for w in range(self.nb_words):
                cnt = X[i][w]
                occs[c][w] += cnt

        self.like = np.zeros((self.nb_classes, self.nb_words))
        for c in range(self.nb_classes):
            for w in range(self.nb_words):
                up = occs[c][w] + self.pseudocount
                down = np.sum(occs[c]) + self.nb_words * self.pseudocount
                self.like[c][w] = up / down

    def predict(self, bow):
        probs = np.zeros(self.nb_classes)
        for c in range(self.nb_classes):
            prob = np.log(self.priors[c])
            for w in range(self.nb_words):
                cnt = bow[w]
                prob += cnt * np.log(self.like[c][w])
            probs[c] = prob
        return np.argmax(probs)


def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    words = text.strip().split()
    cleaned = []
    for word in words:
        if word not in STOPWORDS:
            cleaned.append(word)
    return " ".join(cleaned)


def load_dataset(path):
    texts = []
    labels = []
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = clean_text(row["text"])
            label = int(row["target"])
            texts.append(text)
            labels.append(label)
    return texts, labels


def build_vocab(texts, max_vocab=10000):
    freq = {}
    for text in texts:
        words = text.split()
        for word in words:
            if word in freq:
                freq[word] += 1
            else:
                freq[word] = 1

    sorted_items = sorted(freq.items(), key=lambda item: -item[1])

    vocab = {}
    idx = 0
    for item in sorted_items:
        if idx >= max_vocab:
            break
        vocab[item[0]] = idx
        idx += 1

    return vocab


def texts_to_bow(texts, vocab):
    X = []
    for text in texts:
        bow = []
        for i in range(len(vocab)):
            bow.append(0)
        words = text.split()
        for word in words:
            if word in vocab:
                index = vocab[word]
                bow[index] += 1
        X.append(bow)
    return np.array(X)


def accuracy(model, X_test, Y_test):
    correct = 0
    total = len(X_test)
    for i in range(total):
        prediction = model.predict(X_test[i])
        if prediction == Y_test[i]:
            correct += 1
    return float(correct) / total


def analyze_words(X, Y, vocab_inv):
    pos_counts = np.zeros(len(vocab_inv))
    neg_counts = np.zeros(len(vocab_inv))

    for i in range(len(X)):
        if Y[i] == 1:
            pos_counts += X[i]
        else:
            neg_counts += X[i]

    top_pos = []
    for i in range(len(pos_counts)):
        top_pos.append((i, pos_counts[i]))
    top_pos.sort(key=lambda x: -x[1])

    top_neg = []
    for i in range(len(neg_counts)):
        top_neg.append((i, neg_counts[i]))
    top_neg.sort(key=lambda x: -x[1])

    print("5 najkoriscenijih u pozitivnim:")
    for i in range(5):
        idx = top_pos[i][0]
        print(vocab_inv[idx], int(pos_counts[idx]))

    print("5 najkoriscenijih u negativnim:")
    for i in range(5):
        idx = top_neg[i][0]
        print(vocab_inv[idx], int(neg_counts[idx]))

    # for i in range(len(vocab_inv)):
    #     word = vocab_inv[i]
    #     print(word + ": pos=" + str(pos_counts[i]) + ", neg=" + str(neg_counts[i]))

    lr = []
    for i in range(len(vocab_inv)):
        if pos_counts[i] >= 10 and neg_counts[i] >= 10:
            score = pos_counts[i] / neg_counts[i]
            lr.append((i, score))

    lr_sorted = sorted(lr, key=lambda x: -x[1])

    print("5 najvisih LR:")
    for i in range(5):
        idx, score = lr_sorted[i]
        print(vocab_inv[idx] + ": " + str(round(score, 2)))

    print("5 najmanjih LR:")
    for i in range(5):
        idx, score = lr_sorted[len(lr_sorted) - 1 - i]
        print(vocab_inv[idx] + ": " + str(round(score, 2)))



random.seed()

texts, labels = load_dataset("disaster-tweets.csv")
vocab = build_vocab(texts, max_vocab=10000)

vocab_inv = {}
for word in vocab:
    idx = vocab[word]
    vocab_inv[idx] = word

X = texts_to_bow(texts, vocab)
Y = np.array(labels)

indices = []
for i in range(len(X)):
    indices.append(i)

random.shuffle(indices)

split = int(0.8 * len(X))
train_idx = []
test_idx = []
for i in range(len(X)):
    if i < split:
        train_idx.append(indices[i])
    else:
        test_idx.append(indices[i])

X_train = X[train_idx]
Y_train = Y[train_idx]
X_test = X[test_idx]
Y_test = Y[test_idx]

model = MultinomialNaiveBayes(nb_classes=2, nb_words=len(vocab), pseudocount=1)
model.fit(X_train, Y_train)

acc = accuracy(model, X_test, Y_test)
print("Preciznost: " + str(round(acc, 2)))

analyze_words(X, Y, vocab_inv)

"""
Top pozitivne reci su najcesce termini vezani za katastrofe (disaster, suicide, police), sto je i ocekivano s obzirnom na dataset.
Top negativne reci su vise "normalne" svakodnevne reci, jer se ne radi o katastrofama.

LR meri koliko puta je neka rec zastupljenija u pozitivnim nego u negativnim tvitovima.
Visok LR je vrlo indikativan za katastrofe (fires, fatal), dok dizak LR signalizira normalne, svakodnevne stvari.

LR bolje pokazuje koje reci su karakteristicne za odredjenu klasnu nego jednostavno brojanje.
"""

Preciznost: 0.8
5 najkoriscenijih u pozitivnim:
news 138
disaster 117
california 111
suicide 110
police 107
5 najkoriscenijih u negativnim:
body 113
video 96
people 90
love 90
time 84
5 najvisih LR:
train: 5.64
fires: 5.31
fatal: 4.91
injured: 4.2
latest: 4.0
5 najmanjih LR:
body: 0.11
love: 0.12
night: 0.25
bad: 0.29
fear: 0.3


'\nTop pozitivne reci su najcesce termini vezani za katastrofe (disaster, suicide, police), sto je i ocekivano s obzirnom na dataset.\nTop negativne reci su vise "normalne" svakodnevne reci, jer se ne radi o katastrofama.\n\nLR meri koliko puta je neka rec zastupljenija u pozitivnim nego u negativnim tvitovima.\nVisok LR je vrlo indikativan za katastrofe (fires, fatal), dok dizak LR signalizira normalne, svakodnevne stvari.\n\nLR bolje pokazuje koje reci su karakteristicne za odredjenu klasnu nego jednostavno brojanje.\n'